# Workshop: real-gold 4D-STEM — browse, BF, DF, probe, DPC (Colab T4)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](GIST_PLACEHOLDER)

Load a real 4D-STEM gold dataset from Hugging Face, browse it interactively,
compute bright field / dark field, find the probe center, and visualize DPC —
all in your browser, on Colab's free T4 GPU. No local install, no `quantem.live`.

Two installs only: `quantem.widget` (TestPyPI prerelease) and `quantem`
(from the `berk-workshop` branch on `bobleesj/quantem`). Everything runs on
torch on the GPU.

**Total runtime: 2–3 minutes** (install dominates).

In [ ]:
!pip install -q --pre --extra-index-url https://test.pypi.org/simple/ quantem.widget huggingface_hub
!pip install -q git+https://github.com/bobleesj/quantem.git@berk-workshop

In [ ]:
import quantem as em
import quantem.widget
import torch

print("quantem        ", em.__version__)
print("quantem.widget ", quantem.widget.__version__)
print("torch          ", torch.__version__)
print("cuda available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU runtime)")

In [ ]:
# Download a pre-binned NumPy bundle from the public Hugging Face dataset and
# wrap it as a torch tensor on the GPU. The 4D shape is (scan_row, scan_col,
# k_row, k_col) = (512, 512, 24, 24); detector binned 8x from the original 192x192
# to keep the Colab download small (~300 MB).
import os, json
import numpy as np
from huggingface_hub import snapshot_download

folder = snapshot_download("bobleesj/quantem-data", repo_type="dataset",
                           allow_patterns=["4dstem/gold_512_npy_bin8/*"])
asset = os.path.join(folder, "4dstem", "gold_512_npy_bin8")
data = np.load(os.path.join(asset, "data.npy"))           # (512, 512, 24, 24) uint16
meta = json.load(open(os.path.join(asset, "meta.json")))

device = "cuda" if torch.cuda.is_available() else "cpu"
t = torch.from_numpy(data).to(device)
dset = em.core.datastructures.Dataset4dstem.from_tensor(
    t, sampling=meta["sampling"], units=meta["units"], name=meta["name"],
)
print(f"dataset on {dset.tensor.device}: shape {dset.shape}, dtype {dset.tensor.dtype}")
print(f"sampling {meta['sampling']} {meta['units']}")
print(f"optics: {meta['voltage_kV']} kV, probe {meta['probe_semiangle_mrad']} mrad, CL {meta['camera_length_mm']} mm")

## Step 1 — Browse the 4D-STEM dataset interactively

Drag the scan cursor in the left panel. The CBED on the right updates live.
This is your real-time bright-field / dark-field — pixels INSIDE the central
disk are BF, OUTSIDE are DF. The widget renders in your browser via WebGPU
(works in Chrome on free Colab).

In [ ]:
quantem.widget.Show4DSTEM(dset)

## Step 2 — The probe (mean diffraction pattern)

Average all diffraction patterns over the scan. What's left is the probe — the
shape of the electron beam at the detector. A clean BF disk should appear.

In [ ]:
mean_dp = dset.tensor.to(torch.float32).mean(dim=(0, 1))   # (24, 24) on GPU
print(f"mean_dp shape {tuple(mean_dp.shape)}, device {mean_dp.device}")

em.core.visualization.show_2d(
    mean_dp.cpu().numpy(),
    title="Probe (mean DP)",
    cmap="inferno",
    cbar=True,
)

## Step 3 — Probe center via center-of-mass

Weighted average of the detector coordinates. Should land near the middle of
the detector (~12, 12) for a centered beam — small offset is normal.

In [ ]:
H, W = mean_dp.shape
row = torch.arange(H, device=mean_dp.device, dtype=torch.float32)[:, None]
col = torch.arange(W, device=mean_dp.device, dtype=torch.float32)[None, :]
total = mean_dp.sum()
cy = (mean_dp * row).sum() / total
cx = (mean_dp * col).sum() / total
print(f"probe center: row={cy.item():.3f}, col={cx.item():.3f}   "
      f"(detector {H}x{W}, geometric center ~{H/2:.1f}, ~{W/2:.1f})")

## Step 4 — Bright field + dark field virtual images

Build an aperture mask at the probe center. For every scan position, sum the
pixels inside the disk (BF) and outside (DF). One line of torch, runs on the
GPU in milliseconds.

In [ ]:
BF_RADIUS_PX = 6.0   # ~30 mrad on the bin8 detector

rr, cc = torch.meshgrid(row.squeeze(), col.squeeze(), indexing="ij")
r_from_center = ((rr - cy) ** 2 + (cc - cx) ** 2).sqrt()
bf_mask = (r_from_center <= BF_RADIUS_PX).float()
df_mask = 1.0 - bf_mask

data_f = dset.tensor.to(torch.float32)
bf = (data_f * bf_mask).sum(dim=(-2, -1)).cpu().numpy()
df = (data_f * df_mask).sum(dim=(-2, -1)).cpu().numpy()

em.core.visualization.show_2d(
    [bf, df, mean_dp.cpu().numpy()],
    title=["Bright field", "Dark field", "Probe"],
    cmap=["gray", "gray", "inferno"],
    scalebar=[
        {"sampling": meta["sampling"][0], "units": meta["units"][0]},
        {"sampling": meta["sampling"][0], "units": meta["units"][0]},
        {"sampling": meta["sampling"][2], "units": meta["units"][2]},
    ],
    axsize=(4, 4),
)

## Step 5 — Differential phase contrast (DPC)

Per-scan-position center-of-mass. Each detector pattern's centroid shifts when
local electric fields deflect the probe. The maps below show those shifts as
signed deflection — red/blue around features means real DPC contrast.

Implemented inline as a single torch reduction on the GPU.

In [ ]:
data_f = dset.tensor.to(torch.float32)
H, W = data_f.shape[-2:]
qx = torch.arange(H, device=data_f.device, dtype=torch.float32)[:, None]
qy = torch.arange(W, device=data_f.device, dtype=torch.float32)[None, :]

total_per_dp = data_f.sum(dim=(-2, -1))                   # (scan_r, scan_c)
com_row = (data_f * qx).sum(dim=(-2, -1)) / total_per_dp
com_col = (data_f * qy).sum(dim=(-2, -1)) / total_per_dp

# Detrend (subtract mean) so the colormap is zero-centered on signed deflection.
com_row -= com_row.mean()
com_col -= com_col.mean()

em.core.visualization.show_2d(
    [com_row.cpu().numpy(), com_col.cpu().numpy()],
    title=["DPC — CoM row (qx)", "DPC — CoM col (qy)"],
    cmap="RdBu_r",
    cbar=True,
    scalebar={"sampling": meta["sampling"][0], "units": meta["units"][0]},
    axsize=(4, 4),
)
print(f"CoM row range [{com_row.min().item():.4f}, {com_row.max().item():.4f}] px")
print(f"CoM col range [{com_col.min().item():.4f}, {com_col.max().item():.4f}] px")

## What you just did

1. Loaded real 4D-STEM gold from Hugging Face → torch tensor on the Colab T4.
2. Browsed it interactively with `Show4DSTEM`.
3. Computed the probe (mean DP) and found its center on the detector.
4. Computed BF and DF virtual images with one aperture mask.
5. Computed DPC (per-scan-position CoM) showing electric-field deflection.

Everything ran on the GPU in seconds.

## Try next

- Swap `gold_512_npy_bin8` → `gold_512_npy_bin4` (factor-of-2 finer detector
  sampling at 4× the download).
- Change `BF_RADIUS_PX` to expand/shrink the BF disk.
- Detrend DPC differently (e.g. subtract a plane fit) to reveal smaller features.

v2 will add iterative ptychography (`PtychoLite`) on the same data.